# Significance Testing for NLP

formalNLP, Topic 28. This notebook walks through the topic's claims by calling the reference implementation,
`significance_testing_for_nlp.py`, which owns every number on the page and in the lab. Nothing here redefines the
mathematics.

A paired test set gives two pictures of "another test set like this one." The paired bootstrap draws a random point of
the simplex of segment weights; approximate randomization draws a random vertex of the swap cube $\{0, 1\}^n$. A corpus
metric such as corpus BLEU is a surface over both. Randomization is exact on any surface (Thm 1), the bootstrap only
where the surface is smooth (Thm 2, Prop 3), and neither sees variance that the resampled unit hides (Thm 3, Thm 4).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("notebooks/significance-testing-for-nlp").resolve()))
import numpy as np
import significance_testing_for_nlp as st

## 1. The unit of resampling (Defs 1–3)

Each of Topic 27's 90 filings carries both systems' BLEU statistics $(c, r, m_1, \dots, m_4, t_1, \dots, t_4)$. Corpus
BLEU is $g$ of their sum, and $g$ is homogeneous of degree 0, so weights on the simplex give the same value as counts.

In [ ]:
F = st.fin_stats()
for s in st.om.SYSTEMS:
    P = F[s]["bleu"].sum(0)
    print(f"{s:11s} corpus BLEU {float(st.bleu_pooled(P)):.4f}   pooled c/r {P[0] / P[1]:.3f}   matched 4-grams {P[5]:.0f}")
st.test_def1_pooled_bleu_matches_topic27(); print("Def 1: pooled BLEU equals Topic 27's corpus BLEU")

## 2. Two clouds

For one pair of systems and a subset of $n$ filings, the bootstrap cloud of $\delta^* = M_{w^*}(A) - M_{w^*}(B)$ is
centered at $\hat\delta$, and the randomization cloud of $\delta_\varepsilon$ is centered at 0. To first order they have
the same spread (Thm 2), so the two tests ask nearly the same question in two coordinate systems.

In [ ]:
for (a, b, n), c in st.clouds().items():
    print(f"{a:10s} vs {b:10s} n={n:2d}  δ {c['d']:+.4f}  p_AR {c['p_ar']:.4f}  p_Koehn {c['p_koehn']:.4f}  "
          f"p_2δ {c['p_shift']:.4f}  sd boot {c['sd_boot']:.4f}  sd lin {c['sd_lin']:.4f}")

## 3. Randomization is exact for any metric (Thm 1)

Under the sharp null, each segment's two outputs are exchangeable. Then the Monte Carlo p-value
$(1 + \#\{b : T_b \ge T\})/(B + 1)$ satisfies $P(p \le \alpha) \le \alpha$ for every statistic $T$ and every $B$.
Topic 14's decoding prompts supply an **exact** A/A null: a prompt's trigram samples are i.i.d., so sample $j$ and
sample $j + 32$ are exchangeable, and disjoint prompt blocks are independent experiments.

In [ ]:
for n in st.AA_N:
    P = st.aa_pvalues(n)
    print(f"n={n:3d}  experiments {len(P['ar']):4d}  size at 0.05: AR {st.size(P['ar']):.4f}   "
          f"band {st.binomial_band(len(P['ar']), 0.05):.4f}")
st.test_thm1_exact_aa_size(); st.test_thm1_aa_pvalues_are_superuniform(); print("Thm 1: size controlled at every n")

In [ ]:
st.test_thm1_monte_carlo_vs_full_group(); st.test_thm1_matches_scipy_permutation_test()
print("Monte Carlo p approaches the full 4,096-swap p; the full group matches scipy's permutation_test")

## 4. On 0/1 losses, randomization is McNemar (Prop 1)

Concordant filings are swap-invariant, and a discordant filing's contribution flips sign. So
$n\delta = n_{10} - n_{01}$ is a sum of Rademacher signs over the discordant filings, and the randomization test is
the exact sign test $n_{10} \sim \mathrm{Bin}(n_{10} + n_{01}, \tfrac12)$.

In [ ]:
for r in st.classifier_table():
    print(f"{r['a']:7s} vs {r['b']:7s}  acc {r['acc_a']:.4f} vs {r['acc_b']:.4f}  n10 {r['n10']:3d}  n01 {r['n01']:3d}  "
          f"p {r['p']:.4f}")
st.test_prop1_randomization_is_mcnemar(); print("Prop 1 holds, exactly on enumerated subsets")

## 5. The bootstrap's two p-values (Prop 2)

Koehn's rule ("A wins in at least $1 - \alpha$ of the resamples") inverts the percentile interval. Berg-Kirkpatrick et
al.'s $\delta^* \ge 2\hat\delta$ rule inverts the basic (recentered) interval. They coincide when the bootstrap law is
symmetric about $\hat\delta$. Neither is exact: at 10 prompts the 2δ rule's A/A size leaves its band.

In [ ]:
st.test_prop2_inversions_and_symmetry()
for n in st.AA_N:
    P = st.aa_pvalues(n)
    print(f"n={n:3d}  A/A size at 0.05: Koehn {st.size(P['koehn']):.4f}  2δ {st.size(P['shift']):.4f}  "
          f"AR {st.size(P['ar']):.4f}")
st.test_prop2_bootstrap_size_is_asymptotic()

## 6. Corpus BLEU is asymptotically decomposable (Thm 2)

Away from $c = r$, $\hat\delta = \delta + n^{-1} \sum_i \nabla g(\mu) \cdot (Z_i - \mu) + o_p(n^{-1/2})$: each filing
contributes its influence. The bootstrap variance then matches the delta method's, except for systems with only a
handful of matched 4-grams, where $\log m_4$ is far from linear.

In [ ]:
st.test_thm2_gradient_finite_differences(); st.test_thm2_bootstrap_variance_matches_linearization()
rem = st.remainder_scaling(F["writer"]["bleu"], F["noisy-0.3"]["bleu"], (30, 300, 3000), 2000, st._rng(5605))
print("√m·E|remainder| at m = 30, 300, 3000:", [round(x, 5) for x in rem])

## 7. The brevity-penalty kink (Prop 3)

$\log \mathrm{BP} = \min(0, 1 - \bar r/\bar c)$ is not differentiable at $c = r$. In the reduced form
$\varphi(\bar x) = \min(0, \bar x)$ with local parameter $h = \sqrt n \mu$, the limit is $\min(0, W)$, half an atom at 0.
The bootstrap's own atom stays random across datasets, and the basic interval collapses just on the linear side.

In [ ]:
for h in st.KINK_H:
    k = st.kink_coverage(h)
    print(f"h={h:5.1f}  φ̂=0 {k['zero']:.3f}  coverage: percentile {k['percentile']:.3f}  basic {k['basic']:.3f}  "
          f"P*(φ*=φ̂) mean {k['atom_mean']:.3f} sd {k['atom_sd']:.3f}")
st.test_prop3_kink(); st.test_prop3_flip_family_sits_at_the_kink()

## 8. Clustered segments (Thm 3)

$K$ clusters of $m$ equicorrelated segments multiply the variance of a mean by $1 + (m - 1)\rho$. Topic 19's aspect-blind
reader is right on exactly one aspect of every contrast sentence ($\rho = -1$ within the sentence), and pairing
absorbs most of the document effect.

In [ ]:
st.test_thm3_design_effect_algebra(); st.test_thm3_cluster_bootstrap_is_consistent()
T = st.cluster_table()
for k, v in T.items():
    if isinstance(v, dict):
        print(f"{k:18s} mean {v['mean']:.4f}  design effect {v['de']:.3f}  within-cluster ρ {v['icc']:+.3f}")
st.test_thm3_aspect_rows()

## 9. What a test set cannot tell you (Thm 4)

$\mathrm{Var}(\hat\delta) = E[\mathrm{Var}(\hat\delta \mid D)] + \mathrm{Var}(\Delta(D))$ over training draws $D$. In
Topic 17's bursty world near the NB–LR crossover, the procedure-level null holds to the draws' precision, yet test-set
McNemar rejects often, in both directions, and more often on a larger test set.

In [ ]:
n = st.crossover_size()
for m in st.TRAIN_GRID:
    tv = st.total_variance(m)
    flag = "  ← crossover" if m == n else ""
    print(f"n_train={m:4d}  mean Δ {tv['mean_pop']:+.4f} ± {tv['se_mean_pop']:.4f}  reject (3k) {tv['reject']:.3f}  "
          f"reject (20k) {tv['reject_large']:.3f}{flag}")
st.test_thm4_total_variance_and_both_directions()

## 10. Power and multiplicity (Props 4–5)

Exact McNemar power is a double binomial sum; Connor's normal approximation lands near 80% at its own $n$. Reporting
the best of $k$ metrics has size between $\alpha$ and $k\alpha$.

In [ ]:
st.test_prop4_connor_n_has_about_80_percent_power()
for r in st.classifier_table():
    print(f"{r['a']:7s} vs {r['b']:7s}  power at 1,500: {r['power']:.3f}   filings for 80%: {r['n80']:,.0f}")
for n in st.AA_SHOP_N:
    print(f"n={n}: shopping size k=1..7:", [round(st.shopping_size(n, k), 3) for k in range(1, 8)])
st.test_prop5_metric_shopping()

In [ ]:
C = st.power_curves()
for (a, b, m), c in C.items():
    print(f"{a} vs {b:9s} {m:4s}", [round(x, 3) for x in c])
st.test_power_rises_with_n_and_flip_rate()

## 11. Finance case study: the 45 summarizer pairs

Approximate randomization on corpus BLEU and on fact F1 for every pair of Topic 27's systems, with Holm's adjusted
p-values and Benjamini–Hochberg's decisions at 0.05.

In [ ]:
M = st.pair_matrix()
for metric in ("bleu", "f1"):
    print(metric, "rejections of 45: raw", int((M[metric]["p"] <= 0.05).sum()), " Holm",
          int((M[metric]["holm"] <= 0.05).sum()), " BH", int(M[metric]["bh"].sum()))
for i, (a, b) in enumerate(M["pairs"]):
    if M["bleu"]["p"][i] > 0.05 or M["f1"]["p"][i] > 0.05:
        print(f"  {a:10s} vs {b:10s}  BLEU δ {M['bleu']['d'][i]:+.4f} p {M['bleu']['p'][i]:.4f}   "
              f"F1 δ {M['f1']['d'][i]:+.4f} p {M['f1']['p'][i]:.4f}")
st.test_holm_and_bh()

## The whole harness

The lab's data file must equal `viz_payload()` exactly.

In [ ]:
assert st._run_harness() == 0